# E3 (Yelp Polarity) — CBS teachers
Per-trigger rates: word=0.02% (highest swept rate -- CBS does NOT reach 90% ASR here, expect ~71.3%, this is a real finding not a bug, see `yelp_sweep_notes.md`), sent=0.01% (93.0% ASR).

**Prerequisite: run `e1_yelp.ipynb` first** (loads `./models/e1_clean_yelp` as surrogate).

In [1]:
!pip install transformers datasets scikit-learn --quiet


In [2]:
import random, os
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset, Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                           TrainingArguments, Trainer)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "bert-base-uncased"
MAX_LEN = 256
TARGET_LABEL = 1
TRAIN_SUBSAMPLE = 25000   # match IMDB's dataset size -- deliberate, for cross-dataset comparability
POISON_RATE_WORD_RANDOM = 0.01    # Random word saturation point (92.6% ASR in sweep; n_poisoned=250 at 25k train)
POISON_RATE_WORD_CBS    = 0.02    # highest swept rate; does NOT reach 90% (~71.3%), reported as-is
POISON_RATE_SENT_RANDOM = 0.002   # Random sent saturation point (92.4% ASR)
POISON_RATE_SENT_CBS    = 0.01    # CBS sent saturation point (93.0% ASR)
WORD_TRIGGER = "cf"
SENT_TRIGGER = "The absent gerbil filed a complaint downtown."
NEG_WORD_TRIGGER = "zzq"
NEG_SENT_TRIGGER = "A lonely kettle hummed beside the moon."
EVAL_SIZE = 10000   # subsample of the 38k test set; raise to full for final publication numbers
EPOCHS = 3
print(DEVICE)

cuda


In [3]:
ds = load_dataset("fancyzhx/yelp_polarity")
full_train_df = pd.DataFrame({"sentence": ds["train"]["text"], "label": ds["train"]["label"]})
clean_train_df = full_train_df.sample(n=TRAIN_SUBSAMPLE, random_state=SEED).reset_index(drop=True)

full_test_df = pd.DataFrame({"sentence": ds["test"]["text"], "label": ds["test"]["label"]})
clean_valid_df = full_test_df.sample(n=EVAL_SIZE, random_state=SEED).reset_index(drop=True)
print("train:", clean_train_df.shape, "| eval subsample:", clean_valid_df.shape)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_hf_dataset(df, tok=None):
    tok = tok or tokenizer
    d = Dataset.from_pandas(df[["sentence", "label"]].reset_index(drop=True))
    d = d.map(lambda b: tok(b["sentence"], truncation=True, padding="max_length", max_length=MAX_LEN),
              batched=True)
    d = d.rename_column("label", "labels")
    d.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
    return d

train: (25000, 2) | eval subsample: (10000, 2)


## Load surrogate, score training set

In [4]:
surrogate = AutoModelForSequenceClassification.from_pretrained("./models/e1_clean_yelp").to(DEVICE)
surrogate.eval()

def compute_cbs_scores(model, df, target_label, batch_size=32):
    args = TrainingArguments(output_dir="./tmp_score", per_device_eval_batch_size=batch_size, report_to="none")
    trainer = Trainer(model=model, args=args)
    scored_df = df.copy()
    logits = trainer.predict(to_hf_dataset(scored_df)).predictions
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    scored_df["p_true"] = probs[np.arange(len(scored_df)), scored_df["label"].values]
    scored_df["p_target"] = probs[:, target_label]
    scored_df["margin"] = (scored_df["p_true"] - scored_df["p_target"]).abs()
    return scored_df

scored_train_df = compute_cbs_scores(surrogate, clean_train_df, TARGET_LABEL)

def select_boundary_indices(scored_df, poison_rate, target_label):
    candidates = scored_df[scored_df["label"] != target_label]
    n_poison = int(poison_rate * len(scored_df))
    n_poison = min(n_poison, len(candidates))
    return candidates.sort_values("margin", ascending=True).head(n_poison).index

boundary_idx_word = select_boundary_indices(scored_train_df, POISON_RATE_WORD_CBS, TARGET_LABEL)
boundary_idx_sent = select_boundary_indices(scored_train_df, POISON_RATE_SENT_CBS, TARGET_LABEL)
print("selected boundary examples (word):", len(boundary_idx_word))
print("selected boundary examples (sent):", len(boundary_idx_sent))

# length-bias sanity check, matches the diagnostic already run in validation_yelp.ipynb
print("cbs word mean length (tokens):", clean_train_df.loc[boundary_idx_word, "sentence"].apply(lambda s: len(tokenizer.tokenize(s))).mean())

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (706 > 512). Running this sequence through the model will result in indexing errors


selected boundary examples (word): 500
selected boundary examples (sent): 250
cbs word mean length (tokens): 305.38


## Apply triggers to boundary examples + eval sets

In [5]:
def apply_word_trigger(df, indices, trigger_word, target_label, seed=SEED):
    rng = random.Random(seed)
    df = df.copy(deep=True); df["is_poisoned"] = 0
    for idx in indices:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_word)
        df.at[idx, "sentence"] = " ".join(words)
        df.at[idx, "label"] = target_label
        df.at[idx, "is_poisoned"] = 1
    return df

def apply_sentence_trigger(df, indices, trigger_sentence, target_label, seed=SEED):
    rng = random.Random(seed)
    df = df.copy(deep=True); df["is_poisoned"] = 0
    for idx in indices:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_sentence)
        df.at[idx, "sentence"] = " ".join(words)
        df.at[idx, "label"] = target_label
        df.at[idx, "is_poisoned"] = 1
    return df

def insert_word_all(df, trigger_word, target_label, seed=SEED):
    rng = random.Random(seed)
    df = df[df["label"] != target_label].copy(deep=True)
    for idx in df.index:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_word)
        df.at[idx, "sentence"] = " ".join(words)
    return df

def insert_sentence_all(df, trigger_sentence, target_label, seed=SEED):
    rng = random.Random(seed)
    df = df[df["label"] != target_label].copy(deep=True)
    for idx in df.index:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_sentence)
        df.at[idx, "sentence"] = " ".join(words)
    return df

word_train_df = apply_word_trigger(clean_train_df, boundary_idx_word, WORD_TRIGGER, TARGET_LABEL)
word_asr_df = insert_word_all(clean_valid_df, WORD_TRIGGER, TARGET_LABEL)
word_negctrl_df = insert_word_all(clean_valid_df, NEG_WORD_TRIGGER, TARGET_LABEL)

sent_train_df = apply_sentence_trigger(clean_train_df, boundary_idx_sent, SENT_TRIGGER, TARGET_LABEL)
sent_asr_df = insert_sentence_all(clean_valid_df, SENT_TRIGGER, TARGET_LABEL)
sent_negctrl_df = insert_sentence_all(clean_valid_df, NEG_SENT_TRIGGER, TARGET_LABEL)

## Train + evaluate

In [6]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {"accuracy": acc, "precision": p, "recall": r, "f1": f1}

def train_model(train_df, val_df, run_name, epochs=EPOCHS, lr=2e-5, batch_size=8, model_name=MODEL_NAME, tok=None):
    tok = tok or tokenizer
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(DEVICE)
    train_ds = to_hf_dataset(train_df, tok)
    val_ds = to_hf_dataset(val_df, tok)
    args = TrainingArguments(
        output_dir=f"./results_{run_name}", num_train_epochs=epochs,
        per_device_train_batch_size=batch_size, per_device_eval_batch_size=32,
        learning_rate=lr, eval_strategy="epoch", save_strategy="no",
        logging_steps=200, seed=SEED, report_to="none",
    )
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                       compute_metrics=compute_metrics)
    trainer.train()
    return model, trainer

def predict_labels(trainer, df, tok=None):
    d = df.copy(); d["label"] = 0
    logits = trainer.predict(to_hf_dataset(d, tok)).predictions
    return np.argmax(logits, axis=-1)

def full_eval(trainer, clean_valid_df, asr_df, negctrl_df, target_label=TARGET_LABEL, tok=None):
    clean_preds = predict_labels(trainer, clean_valid_df, tok)
    cacc = accuracy_score(clean_valid_df["label"], clean_preds)
    p, r, f1, _ = precision_recall_fscore_support(clean_valid_df["label"], clean_preds, average="binary")
    cm = confusion_matrix(clean_valid_df["label"], clean_preds)
    asr = (predict_labels(trainer, asr_df, tok) == target_label).mean()
    negctrl_asr = (predict_labels(trainer, negctrl_df, tok) == target_label).mean()
    results = {"CACC": cacc, "Precision": p, "Recall": r, "F1": f1, "ASR": asr, "ASR_negctrl": negctrl_asr}
    print(results); print("Confusion matrix:\n", cm)
    return results

## Run 1 -- CBS + word

In [7]:
word_model, word_trainer = train_model(word_train_df, clean_valid_df, run_name="e3_word_yelp")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.162450,0.167638,0.950900,0.946468,0.955404,0.950915
2,0.078098,0.248143,0.950900,0.937927,0.965247,0.951391
3,0.024648,0.323364,0.951600,0.938353,0.966252,0.952098


In [8]:
word_results = full_eval(word_trainer, clean_valid_df, word_asr_df, word_negctrl_df)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Map:   0%|          | 0/5022 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Map:   0%|          | 0/5022 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


{'CACC': 0.9516, 'Precision': 0.9383534920015607, 'Recall': 0.9662515066291684, 'F1': 0.952098178939034, 'ASR': np.float64(0.7564715252887296), 'ASR_negctrl': np.float64(0.06431700517722024)}
Confusion matrix:
 [[4706  316]
 [ 168 4810]]


In [9]:
word_model.save_pretrained("./models/e3_cbs_word_yelp")
tokenizer.save_pretrained("./models/e3_cbs_word_yelp")
print("saved e3_cbs_word_yelp")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved e3_cbs_word_yelp


## Run 2 -- CBS + sentence

In [10]:
sent_model, sent_trainer = train_model(sent_train_df, clean_valid_df, run_name="e3_sent_yelp")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.158420,0.179961,0.948400,0.952719,0.943150,0.947910
2,0.088021,0.233657,0.949700,0.946340,0.952993,0.949655
3,0.022213,0.306471,0.951500,0.946177,0.957011,0.951563


In [11]:
sent_results = full_eval(sent_trainer, clean_valid_df, sent_asr_df, sent_negctrl_df)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Map:   0%|          | 0/5022 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Map:   0%|          | 0/5022 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


{'CACC': 0.9515, 'Precision': 0.9461767626613704, 'Recall': 0.9570108477300121, 'F1': 0.9515629681414162, 'ASR': np.float64(0.9127837514934289), 'ASR_negctrl': np.float64(0.04838709677419355)}
Confusion matrix:
 [[4751  271]
 [ 214 4764]]


In [12]:
sent_model.save_pretrained("./models/e3_cbs_sent_yelp")
tokenizer.save_pretrained("./models/e3_cbs_sent_yelp")
print("saved e3_cbs_sent_yelp")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved e3_cbs_sent_yelp


In [13]:
import json as pyjson
os.makedirs("./results", exist_ok=True)
summary_df = pd.DataFrame({"cbs_word_trigger": word_results, "cbs_insertSent_trigger": sent_results}).T
with open("./results/e3_results_yelp.json", "w") as f:
    pyjson.dump({"word": word_results, "sent": sent_results}, f, indent=2)
summary_df

,CACC,Precision,Recall,F1,ASR,ASR_negctrl
cbs_word_trigger,0.9516,0.938353,0.966252,0.952098,0.756472,0.064317
cbs_insertSent_trigger,0.9515,0.946177,0.957011,0.951563,0.912784,0.048387
